                    Project Milestone 4:Connecting to an API/Pulling in the Data and Cleaning/Formatting
                    ______________________________________________________________________________________
                                        Name = Komal Shahid
                                        DSC 540
                            

 **import all the libraries**
______________________________

In [1]:
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
import zipfile
import os
import pandas as pd
import re
import numpy as np

**Kaggle API to extract the dataset**
_________________________________________

In [2]:
api = KaggleApi()
api.authenticate()

In [3]:
api.competition_list_files_cli(competition= 'salary-prediction-for-job-postings')

name                           size  creationDate         
----------------------------  -----  -------------------  
usjobs_train.csv                8MB  2023-11-08 14:12:23  
usjobs_test.csv                 5MB  2023-11-08 14:12:23  
usjobs_sample_submission.csv  627KB  2023-11-08 14:12:23  


In [4]:
api.competition_download_file('salary-prediction-for-job-postings', file_name= 'usjobs_train.csv')


usjobs_train.csv.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
def unzip_file(file_path, extract_path):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

unzip_file('usjobs_train.csv.zip', ".")

**Read in the Data set for us jobs**
_____________________________________

In [6]:
df = pd.read_csv('usjobs_train.csv')
df.head()

,ID,Job,Jobs_Group,Profile,Remote,Company,Location,City,State,Frecuency_Salary,...,Skills,Sector,Sector_Group,Revenue,Employee,Company_Score,Reviews,Director,Director_Score,URL
0,job_f2c807527f687b96,"Part-time Reporting Business Analyst, Data & A...",Financial Analyst,NaN,Remote,Sandy Hook Promise Foundation,Remote,NaN,NaN,hour,...,"['Salesforce', 'Bachelor']",NGOs and Nonprofit Organizations,Nonprofit Organizations,NaN,XS,4.2,20.0,NaN,NaN,https://www.sandyhookpromise.org/
1,job_2660d4c53505af10,Controller,Controller,NaN,NaN,Building Service 32BJ Benefit Funds,"New York, NY 10013 (Tribeca area)",New York,NY,year,...,"['SQL', 'Master', 'Dynamics 365', 'Snowflake',...",NGOs and Nonprofit Organizations,Nonprofit Organizations,NaN,M,3.5,58.0,"Peter Goldberger, Executive Director",0.70,NaN
2,sj_50358c44328ae06a,Sr Finance Analyst,Financial Analyst,Senior,NaN,LCS,NaN,NaN,NaN,year,...,"['Word', 'Bachelor', 'Excel']",Personal Consumer Services,Sales,XXXS,XXXS,3.4,88.0,NaN,NaN,NaN
3,job_a087fd700e3e85f0,Senior Business Intelligence Analyst,Business Intelligence,Senior,Hybrid,Federal Reserve Bank of Richmond,"Richmond, VA 23219 (Central Office area)",Richmond,VA,year,...,"['PowerPoint', 'Power BI', 'Tableau', 'Word', ...",Banking and Credit Services,Finance,XXL,XL,3.8,30.0,Tom Barkin,0.70,https://www.richmondfed.org/
4,job_d2a2538a2c4d2033,Data Center Operations Analyst (Temporary Assi...,Operations Analyst,NaN,Remote,Los Angeles County Office of Education,"Downey, CA 90242+1 ubicación",Downey,CA,hour,...,['Office'],State and Regional Agencies,Government,NaN,XL,4.2,186.0,Debra Duardo,0.85,NaN


In [7]:
print(df.shape)

(33248, 21)


                                Step 1: Duplicates(Unique Values) and Null values
                                ___________________________________________________

In [8]:
for column in df.columns:
    unique_values = len(df[column].value_counts())
    null_values = df[column].isna().sum()
    
    print(f"{column} Total Unique Values: {unique_values}")
    print(f"{column} Null Values Count: {null_values}")
    print()

ID Total Unique Values: 33248
ID Null Values Count: 0

Job Total Unique Values: 17227
Job Null Values Count: 0

Jobs_Group Total Unique Values: 14
Jobs_Group Null Values Count: 0

Profile Total Unique Values: 3
Profile Null Values Count: 21107

Remote Total Unique Values: 2
Remote Null Values Count: 19319

Company Total Unique Values: 13995
Company Null Values Count: 9

Location Total Unique Values: 12542
Location Null Values Count: 13

City Total Unique Values: 2951
City Null Values Count: 3824

State Total Unique Values: 54
State Null Values Count: 3112

Frecuency_Salary Total Unique Values: 5
Frecuency_Salary Null Values Count: 0

Mean_Salary Total Unique Values: 8454
Mean_Salary Null Values Count: 0

Skills Total Unique Values: 10805
Skills Null Values Count: 0

Sector Total Unique Values: 138
Sector Null Values Count: 7214

Sector_Group Total Unique Values: 28
Sector_Group Null Values Count: 7214

Revenue Total Unique Values: 9
Revenue Null Values Count: 18318

Employee Total Uniq

                                    Step 2:   drop columns with null values
                                ___________________________________________________

In [9]:

null_threshold = len(df) * 0.2  # Set the threshold as 50% of total rows
columns_to_drop = []
for column in df.columns:
    null_values = df[column].isna().sum()
    
    if null_values > null_threshold:
        columns_to_drop.append(column)

df.drop(columns=columns_to_drop, inplace=True)

print(f"Columns dropped: {columns_to_drop}")

Columns dropped: ['Profile', 'Remote', 'Sector', 'Sector_Group', 'Revenue', 'Employee', 'Company_Score', 'Reviews', 'Director', 'Director_Score', 'URL']


In [10]:
df.isna().sum()


ID                     0
Job                    0
Jobs_Group             0
Company                9
Location              13
City                3824
State               3112
Frecuency_Salary       0
Mean_Salary            0
Skills                 0
dtype: int64

                                        Step 3:  Fix casing or inconsistent values
                                        ____________________________________________

In [11]:
#  Extract job title
df['Job'] = df['Job'].apply(lambda x: re.split('[,(]', x)[0].strip())
df

,ID,Job,Jobs_Group,Company,Location,City,State,Frecuency_Salary,Mean_Salary,Skills
0,job_f2c807527f687b96,Part-time Reporting Business Analyst,Financial Analyst,Sandy Hook Promise Foundation,Remote,NaN,NaN,hour,115000.000,"['Salesforce', 'Bachelor']"
1,job_2660d4c53505af10,Controller,Controller,Building Service 32BJ Benefit Funds,"New York, NY 10013 (Tribeca area)",New York,NY,year,185000.000,"['SQL', 'Master', 'Dynamics 365', 'Snowflake',..."
2,sj_50358c44328ae06a,Sr Finance Analyst,Financial Analyst,LCS,NaN,NaN,NaN,year,84500.000,"['Word', 'Bachelor', 'Excel']"
3,job_a087fd700e3e85f0,Senior Business Intelligence Analyst,Business Intelligence,Federal Reserve Bank of Richmond,"Richmond, VA 23219 (Central Office area)",Richmond,VA,year,111625.000,"['PowerPoint', 'Power BI', 'Tableau', 'Word', ..."
4,job_d2a2538a2c4d2033,Data Center Operations Analyst,Operations Analyst,Los Angeles County Office of Education,"Downey, CA 90242+1 ubicación",Downey,CA,hour,102690.400,['Office']
...,...,...,...,...,...,...,...,...,...,...
33243,job_51375f2d25e41629,Accounting Specialist I - Treasurer,Finance,Yavapai County Human Resources,"Prescott, AZ",Prescott,AZ,year,47206.495,"['English', 'Office']"
33244,job_af259ef51491c1fd,Treasurer Accounting Supervisor,Finance,Pinal County,"Florence, AZ 85132",Florence,AZ,year,79741.000,"['CPA', 'Office', 'Bachelor']"
33245,job_45519b91f601ab4c,FINANCIAL MANAGEMENT ANALYST,Financial Analyst,US Naval Air Systems Command,"Patuxent River, MD+1 ubicación",Patuxent River,MD,year,119908.000,"['Office', 'Bachelor']"
33246,job_25a66c0757505368,BA with Management of Change,Business Analyst,Source Mantra,"Houston, TX 77015",Houston,TX,hour,115000.000,['SQL']


In [12]:
#  Extract individual skills separated by commas
df['Skills'] = df['Skills'].apply(lambda x: x.strip(" '' "))
df

,ID,Job,Jobs_Group,Company,Location,City,State,Frecuency_Salary,Mean_Salary,Skills
0,job_f2c807527f687b96,Part-time Reporting Business Analyst,Financial Analyst,Sandy Hook Promise Foundation,Remote,NaN,NaN,hour,115000.000,"['Salesforce', 'Bachelor']"
1,job_2660d4c53505af10,Controller,Controller,Building Service 32BJ Benefit Funds,"New York, NY 10013 (Tribeca area)",New York,NY,year,185000.000,"['SQL', 'Master', 'Dynamics 365', 'Snowflake',..."
2,sj_50358c44328ae06a,Sr Finance Analyst,Financial Analyst,LCS,NaN,NaN,NaN,year,84500.000,"['Word', 'Bachelor', 'Excel']"
3,job_a087fd700e3e85f0,Senior Business Intelligence Analyst,Business Intelligence,Federal Reserve Bank of Richmond,"Richmond, VA 23219 (Central Office area)",Richmond,VA,year,111625.000,"['PowerPoint', 'Power BI', 'Tableau', 'Word', ..."
4,job_d2a2538a2c4d2033,Data Center Operations Analyst,Operations Analyst,Los Angeles County Office of Education,"Downey, CA 90242+1 ubicación",Downey,CA,hour,102690.400,['Office']
...,...,...,...,...,...,...,...,...,...,...
33243,job_51375f2d25e41629,Accounting Specialist I - Treasurer,Finance,Yavapai County Human Resources,"Prescott, AZ",Prescott,AZ,year,47206.495,"['English', 'Office']"
33244,job_af259ef51491c1fd,Treasurer Accounting Supervisor,Finance,Pinal County,"Florence, AZ 85132",Florence,AZ,year,79741.000,"['CPA', 'Office', 'Bachelor']"
33245,job_45519b91f601ab4c,FINANCIAL MANAGEMENT ANALYST,Financial Analyst,US Naval Air Systems Command,"Patuxent River, MD+1 ubicación",Patuxent River,MD,year,119908.000,"['Office', 'Bachelor']"
33246,job_25a66c0757505368,BA with Management of Change,Business Analyst,Source Mantra,"Houston, TX 77015",Houston,TX,hour,115000.000,['SQL']


In [13]:
# Location remove 

In [14]:
df['Location'] = df['Location'].dropna().apply(lambda x: re.split('[(+]', x)[0].strip())
df

,ID,Job,Jobs_Group,Company,Location,City,State,Frecuency_Salary,Mean_Salary,Skills
0,job_f2c807527f687b96,Part-time Reporting Business Analyst,Financial Analyst,Sandy Hook Promise Foundation,Remote,NaN,NaN,hour,115000.000,"['Salesforce', 'Bachelor']"
1,job_2660d4c53505af10,Controller,Controller,Building Service 32BJ Benefit Funds,"New York, NY 10013",New York,NY,year,185000.000,"['SQL', 'Master', 'Dynamics 365', 'Snowflake',..."
2,sj_50358c44328ae06a,Sr Finance Analyst,Financial Analyst,LCS,NaN,NaN,NaN,year,84500.000,"['Word', 'Bachelor', 'Excel']"
3,job_a087fd700e3e85f0,Senior Business Intelligence Analyst,Business Intelligence,Federal Reserve Bank of Richmond,"Richmond, VA 23219",Richmond,VA,year,111625.000,"['PowerPoint', 'Power BI', 'Tableau', 'Word', ..."
4,job_d2a2538a2c4d2033,Data Center Operations Analyst,Operations Analyst,Los Angeles County Office of Education,"Downey, CA 90242",Downey,CA,hour,102690.400,['Office']
...,...,...,...,...,...,...,...,...,...,...
33243,job_51375f2d25e41629,Accounting Specialist I - Treasurer,Finance,Yavapai County Human Resources,"Prescott, AZ",Prescott,AZ,year,47206.495,"['English', 'Office']"
33244,job_af259ef51491c1fd,Treasurer Accounting Supervisor,Finance,Pinal County,"Florence, AZ 85132",Florence,AZ,year,79741.000,"['CPA', 'Office', 'Bachelor']"
33245,job_45519b91f601ab4c,FINANCIAL MANAGEMENT ANALYST,Financial Analyst,US Naval Air Systems Command,"Patuxent River, MD",Patuxent River,MD,year,119908.000,"['Office', 'Bachelor']"
33246,job_25a66c0757505368,BA with Management of Change,Business Analyst,Source Mantra,"Houston, TX 77015",Houston,TX,hour,115000.000,['SQL']


                                            Step 4 : Extract state and zip code from location
                                            __________________________________________

In [15]:
# Takeout the state from location
df['State_Extracted'] = df['Location'].dropna().apply(lambda x: re.search(r'(\w{2})\s*\d{5}', x).group(1) if re.search(r'(\w{2})\s*\d{5}', x) else '')
df['State_Extracted'].dropna()
df.drop(columns=['State','Frecuency_Salary'], inplace=True)

                                                 Step 5:Identify outliers and bad data
                                                ________________________________________

In [16]:
# Identify outliers in salary column
z_scores = (df['Mean_Salary'].astype('int64') - df['Mean_Salary'].astype('int64').mean()) / df['Mean_Salary'].astype('int64').std()
threshold = 3  # Adjust the threshold if desired
outliers = df[np.abs(z_scores) > threshold]
outliers

,ID,Job,Jobs_Group,Company,Location,City,Mean_Salary,Skills,State_Extracted
187,job_f249a027413b6df0,Engineering Manager,ML/AI Engineer,"Grammarly, Inc.",United States,NaN,370000.0,"['Machine Learning', 'Artificial Intelligence']",
220,job_aa14df6887a2c79c,Chief Financial Officer,CFO,Pro Talent Solutions,"Brooklyn, NY 11221",Brooklyn,250000.0,"['CPA', 'Office', 'Bachelor']",NY
301,job_3ad0a42fcc2364ec,Director - US Treasury Capital Markets Counsel,CFO,Barclays,"New York, NY 10020",New York,295000.0,['Office'],NY
384,job_4abfc9929cb3aba3,Senior/ Staff Machine Learning Research Scientist,ML/AI Engineer,Nuro,"Mountain View, CA",Mountain View,252700.0,"['Python', 'C++', 'Deep Learning', 'Machine Le...",
556,job_1986fa39e76c9d03,Assistant Controller,Controller,Freshworks,"San Mateo, CA",San Mateo,254657.5,"['CPA', 'Bachelor', 'ERP']",
...,...,...,...,...,...,...,...,...,...
32617,job_6f283515f01d4b1a,Vice President and Chief Financial Officer,CFO,"AA Transportation Co., Inc.","Nashville, TN",Nashville,408450.0,"['CPA', 'Office']",
32668,job_202707d0613505dc,Machine Learning Scientist - Natural Language ...,ML/AI Engineer,JPMorgan Chase & Co,"New York, NY 10179",New York,262500.0,"['Tensor Flow', 'PhD', 'SciKit', 'Pandas', 'De...",NY
32728,job_25f8c0d279153a3d,Enterprise VP,Financial Analyst,Lincoln Financial,"Radnor, PA",Radnor,237950.5,"['PowerPoint', 'MBA', 'Excel', 'CPA', 'Bachelo...",
32809,job_9a6c44709bafe0f5,Sr. Machine Learning Engineer,ML/AI Engineer,Braintrust,"Austin, TX",Austin,325000.0,"['Tensor Flow', 'PhD', 'Machine Learning', 'Ar...",
